# 03 EDA & Feature Selection

Responsibility
--------------
02 Engineered Learning Dataset을 입력으로 받아:

  1. Generic Validation
  2. Dataset / Data Quality EDA
  3. Feature Candidate / Target Candidate 분류
  4. Feature EDA
  5. Feature Quality Metrics
  6. Feature Redundancy Analysis
  7. Feature ↔ Target Correlation Analysis
  8. Bearing-wise Feature ↔ Target Analysis
  9. EDA Evidence 저장
 10. 사용자 명시적 Feature / Target Selection

Pipeline
--------
```
02 Engineered Learning Dataset
           ↓
     Generic Validation
           ↓
      Column Classification
           ↓
      ┌──── EDA ────┐
      │             │
      ▼             ▼
Feature Evidence  Target Evidence
      │             │
      └──────┬──────┘
             ▼
       Evidence Saving
             ↓
       User Decision
             ↓
    CANDIDATE_FEATURES
    SELECTED_TARGET
```

This Notebook does NOT perform:
  - Feature Transformation
  - RUL Target Generation
  - Automatic Feature Selection
  - Automatic Target Selection
  - Scaling
  - Sequence / Windowing
  - Dataset Builder
  - Test EDA

Important
---------
Feature Ranking is Evidence only.
Ranking score does not automatically determine selection.

Rectified_* and RDI_* are both Feature Candidates.

failure_time is an auxiliary calculation column and is not:
  - Metadata
  - Target Candidate
  - Feature Candidate

In [ ]:
# =============================================================================
# 0. Imports
# =============================================================================
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils.project_utils import *
from utils.bearing_utils import META_COLUMNS, TARGET_CANDIDATES
from utils.eda_utils import *

warnings.filterwarnings("ignore")

# Visualization defaults
plt.style.use("default")
sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False

# Display defaults
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


# =============================================================================
# 1. Path Configuration
# =============================================================================
LEARNING_INPUT_PATH = "outputs/02_feature_transformation/learning_engineered_features.csv"
EVIDENCE_DIR = "outputs/03_eda_selection/eda_evidence"

FEATURE_METRICS_PATH = os.path.join(EVIDENCE_DIR, "feature_metrics.csv")
FEATURE_REDUNDANCY_PATH = os.path.join(EVIDENCE_DIR, "feature_redundancy.csv")

RUL_CORRELATION_PATH = os.path.join(EVIDENCE_DIR, "rul_correlation.csv")
RUL_CORRELATION_BY_BEARING_PATH = os.path.join(EVIDENCE_DIR, "rul_correlation_by_bearing.csv")

DATA_QUALITY_PATH = os.path.join(EVIDENCE_DIR, "data_quality.csv")
BEARING_PROFILE_PATH = os.path.join(EVIDENCE_DIR, "bearing_profile.csv")

# Visualization output directory
FIGURE_DIR = os.path.join(EVIDENCE_DIR, "figures")


# =============================================================================
# 2. EDA Configuration
# =============================================================================
#
# Visualization 대상은 분석 목적에 따라 분리한다.
#
# Non-cumulative Features
#   - Distribution
#   - Operating Condition
#   - Bearing Degradation Trend
#
# Cumulative Features
#   - RUL Correlation
#   - Optional Feature ↔ Target Visualization
#
# 주의:
#   이 목록들은 Feature Selection 결과가 아니다.
#   실제 Feature Candidate 전체는 이후 feature_candidates로 자동 분류한다.
#
# -------------------------------------------------------------------------
# Non-cumulative Feature Visualization
# -------------------------------------------------------------------------
#
# Raw Feature와 Rectified Feature를 대상으로 한다.
#
NON_CUMULATIVE_FEATURES = ["rms", "cf"]

# -------------------------------------------------------------------------
# Cumulative Feature Visualization
# -------------------------------------------------------------------------
#
# 현재 Transformation에서 최종적으로 생성되는 누적 계열인 RDI를 대상으로 한다.
#
CUMULATIVE_FEATURES = ["RDI_rms", "RDI_cf"]

# -------------------------------------------------------------------------
# RUL Target for Visualization
# -------------------------------------------------------------------------
RUL_VISUALIZATION_TARGET = "RUL_norm"

# -------------------------------------------------------------------------
# Detailed Feature ↔ Target Plot
# -------------------------------------------------------------------------
#
# 빈 리스트이면 상세 산점도를 생성하지 않는다.
#
# 필요할 경우 Evidence를 검토한 뒤 직접 지정한다.
#
RUL_PLOT_FEATURES = CUMULATIVE_FEATURES

# -------------------------------------------------------------------------
# Redundancy Evidence threshold
# -------------------------------------------------------------------------
REDUNDANCY_THRESHOLD = 0.85

# -------------------------------------------------------------------------
# Ranking weights
# -------------------------------------------------------------------------
RANKING_WEIGHTS = {
    "Monotonicity": 0.35,
    "Trendability": 0.35,
    "Prognosability": 0.20,
    "CV": 0.10,
}

# -------------------------------------------------------------------------
# Evidence output
# -------------------------------------------------------------------------
SAVE_EVIDENCE = True

# Visualization output
SAVE_FIGURES = True

In [ ]:
# =============================================================================
# 3. Dataset Loading
# =============================================================================
if not os.path.exists(LEARNING_INPUT_PATH):
    raise FileNotFoundError(
        "Engineered Learning Dataset not found: "
        f"{LEARNING_INPUT_PATH}"
    )

df = pd.read_csv(LEARNING_INPUT_PATH)

print("=" * 60)
print("Engineered Learning Dataset Loaded")
print("=" * 60)

print(f"Path    : {LEARNING_INPUT_PATH}")

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

display(df.head(3))

In [ ]:
# =============================================================================
# 4. Generic Validation
# =============================================================================
#
# 03 입력은 02에서 생성된 Engineered Learning Dataset이다.
#
# 02에서 이미 Target이 생성되었으므로,
# 여기서는 Target Generation을 수행하지 않는다.
#
print("=" * 60)
print("Generic Validation")
print("=" * 60)

# -------------------------------------------------------------------------
# Required Metadata
# -------------------------------------------------------------------------
assert_required_columns(df, META_COLUMNS)

# -------------------------------------------------------------------------
# Required Target Candidates
# -------------------------------------------------------------------------
assert_required_columns(df, TARGET_CANDIDATES)

# -------------------------------------------------------------------------
# Time Ordering
# -------------------------------------------------------------------------
df = sort_by_time(
    df, bearing_col="bearing", time_col="elapsed_sec"
)

assert_monotonic_order(
    df, group_col="bearing", order_col="elapsed_sec"
)

print("Required metadata columns : OK")
print("Target candidate columns  : OK")
print("Temporal ordering         : OK")

In [ ]:
# =============================================================================
# 5. Dataset Structural Overview
# =============================================================================

print("=" * 60)
print("Dataset Structural Overview")
print("=" * 60)

print(f"Rows          : {len(df):,}")
print(f"Columns       : {len(df.columns):,}")

print(
    "Memory Usage  : "
    f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

print(f"Bearings      : {df['bearing'].nunique():,}")


# Column listing

print("\nColumns:")
for column in df.columns:
    print(f"  - {column}")

In [ ]:
# =============================================================================
# 6. Column Classification
# =============================================================================
#
# Semantic classification:
#
# Metadata
#   bearing
#   condition
#   rpm
#   load
#   file_idx
#   hour
#   minute
#   second
#   elapsed_sec
#
# Target Candidates
#   life_ratio
#   RUL_sec
#   RUL_norm
#
# Feature Candidates
#   Everything else
#   except failure_time
#
# Rectified_* / RDI_* are included as Feature Candidates.
#
metadata_columns, remaining_columns = split_columns(df, META_COLUMNS)

target_columns = [
    column for column in TARGET_CANDIDATES
    if column in remaining_columns
]

feature_candidates = [
    column for column in remaining_columns
    if column not in TARGET_CANDIDATES
    and column != "failure_time"
]

# -------------------------------------------------------------------------
# Candidate integrity check
# -------------------------------------------------------------------------
non_numeric_features = [
    column for column in feature_candidates
    if not pd.api.types.is_numeric_dtype(df[column])
]

if non_numeric_features:
    raise TypeError(
        "Feature Candidates must be numeric for EDA metrics. "
        f"Non-numeric columns: {non_numeric_features}"
    )


print("=" * 60)
print("Column Classification")
print("=" * 60)

print(f"Metadata columns        : {len(metadata_columns):,}")
print(f"Target Candidates       : {len(target_columns):,}")
print(f"Feature Candidates      : {len(feature_candidates):,}")

print(
    "Auxiliary columns       : "
    f"{['failure_time'] if 'failure_time' in df.columns else []}"
)

print("\nTarget Candidates:")
for column in target_columns:
    print(f"  - {column}")

print("\nFeature Candidates:")
for column in feature_candidates:
    print(f"  - {column}")

In [ ]:
# =============================================================================
# 7. Data Integrity Check
# =============================================================================

print("=" * 60)
print("Data Integrity Check")
print("=" * 60)


# -------------------------------------------------------------------------
# Duplicate Rows
# -------------------------------------------------------------------------

duplicate_count = int(
    df.duplicated().sum()
)

print(
    f"Duplicate Rows : {duplicate_count:,}"
)


# -------------------------------------------------------------------------
# Infinite Values
# -------------------------------------------------------------------------

numeric_columns = (
    df.select_dtypes(include=np.number)
    .columns
)

inf_summary = {}

for column in numeric_columns:
    count = int(
        np.isinf(df[column]).sum()
    )

    if count > 0:
        inf_summary[column] = count


if inf_summary:
    print("\nInfinite Values:")

    for column, count in inf_summary.items():
        print(
            f"  - {column}: {count:,}"
        )

else:
    print(
        "Infinite Values : none"
    )


# -------------------------------------------------------------------------
# Missing Values
# -------------------------------------------------------------------------
#
# 결측이 실제로 존재하는 컬럼만 표시한다.
# 결측이 전혀 없으면 표를 만들지 않고 상태만 출력한다.
#

missing_df = (
    pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing Ratio (%)": (
            df.isna().mean() * 100
        ),
    })
    .query("`Missing Count` > 0")
    .sort_values(
        "Missing Ratio (%)",
        ascending=False,
    )
)


print("\nMissing Values:")

if missing_df.empty:
    print("  None")

else:
    display(missing_df)


# -------------------------------------------------------------------------
# Data Quality Evidence
# -------------------------------------------------------------------------
#
# Evidence 파일에는 전체 컬럼에 대한 정보를 보존한다.
# 단, 화면에는 위의 Missing Value Summary만 표시한다.
#

data_quality = (
    pd.DataFrame({
        "Column": df.columns,
        "Missing Count": [
            int(df[column].isna().sum())
            for column in df.columns
        ],
        "Missing Ratio (%)": [
            float(df[column].isna().mean() * 100)
            for column in df.columns
        ],
    })
)

data_quality["Duplicate_Rows_Total"] = (
    duplicate_count
)

data_quality["Infinite_Count"] = (
    data_quality["Column"]
    .map(inf_summary)
    .fillna(0)
    .astype(int)
)

In [ ]:
# =============================================================================
# 8. Bearing Run-to-Failure Profile
# =============================================================================
print("=" * 60)
print("Bearing Run-to-Failure Profile")
print("=" * 60)

bearing_profile = (
    df.groupby("bearing").agg(
        Samples=("elapsed_sec", "count"),
        Max_Elapsed_Sec=("elapsed_sec", "max"),
        Condition=("condition", "first"),
        RPM=("rpm", "first"),
        Load=("load", "first"),
    ).reset_index()
)

display(bearing_profile)

# -------------------------------------------------------------------------
# Bearing Profile Visualization
# -------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(
    x="bearing",
    y="Samples",
    data=bearing_profile,
    ax=axes[0]
)

axes[0].set_title("Samples per Bearing")
axes[0].tick_params(axis="x", rotation=30)


sns.countplot(
    x="condition",
    data=df,
    ax=axes[1],
    order=sorted(df["condition"].dropna().unique())
)

axes[1].set_title("Samples per Operating Condition")

plt.tight_layout()
plt.show()

if SAVE_FIGURES:
    os.makedirs(FIGURE_DIR, exist_ok=True)

    fig.savefig(
        os.path.join(FIGURE_DIR, "bearing_profile.png"),
        dpi=150, bbox_inches="tight"
    )

In [ ]:
# =============================================================================
# 9. Feature Distribution EDA
# =============================================================================
#
# Distribution은 Non-cumulative Feature를 대상으로 한다.
#

available_non_cumulative = [
    feature for feature in NON_CUMULATIVE_FEATURES
    if feature in feature_candidates
]

missing_non_cumulative = [
    feature for feature in NON_CUMULATIVE_FEATURES
    if feature not in feature_candidates
]


print("=" * 60)
print("Feature Distribution EDA")
print("=" * 60)


if missing_non_cumulative:
    print("Visualization target not available:")

    for feature in missing_non_cumulative:
        print(f"  - {feature}")


for feature in available_non_cumulative:
    print(f"\n[Distribution] {feature}")

    fig = plot_distribution(df, feature)

    if SAVE_FIGURES:
        fig.savefig(
            os.path.join(FIGURE_DIR, f"distribution_{feature}.png"),
            dpi=150, bbox_inches="tight",
        )

In [ ]:
# =============================================================================
# 10. Operating Condition EDA
# =============================================================================
#
# Operating Condition 분석도 Non-cumulative Feature를 대상으로 한다.
#

print("=" * 60)
print("Operating Condition EDA")
print("=" * 60)


for feature in available_non_cumulative:
    print(f"\n[Condition] {feature}")
    fig = plot_condition(df, feature, condition_col="condition")

    if SAVE_FIGURES:
        fig.savefig(
            os.path.join(FIGURE_DIR, f"condition_{feature}.png"),
            dpi=150, bbox_inches="tight",
        )

In [ ]:
# =============================================================================
# 11. Bearing Degradation Trend EDA
# =============================================================================
#
# Bearing 간 열화 진행도를 비교하기 위해 life_ratio를 공통 x축으로 사용한다.
#
# elapsed_sec 기반 그래프는 중복 정보로 판단하여 생성하지 않는다.
#

print("=" * 60)
print("Bearing Degradation Trend EDA")
print("=" * 60)


for feature in available_non_cumulative:
    print(f"\n[Life Ratio] {feature}")

    fig = plot_bearing_trend(
        df, feature,
        x_col="life_ratio",
        bearing_col="bearing",
    )

    if SAVE_FIGURES:
        fig.savefig(
            os.path.join(FIGURE_DIR, f"trend_life_ratio_{feature}.png"),
            dpi=150, bbox_inches="tight",
        )

In [ ]:
# =============================================================================
# 12. Feature Quality Metrics
# =============================================================================
#
# Metrics:
#   - Monotonicity
#   - Trendability
#   - Prognosability
#   - Condition Stability (CV)
#   - Composite Ranking Score
#
# Ranking is Evidence only.
# It does not determine Feature Selection.
#
print("=" * 60)
print("Feature Quality Metrics")
print("=" * 60)

feature_metrics = rank_features(
    df, feature_candidates,
    weights=RANKING_WEIGHTS,
)

display(feature_metrics)


# -------------------------------------------------------------------------
# Top Ranked Features
# -------------------------------------------------------------------------
print("=" * 60)
print("Top Ranked Features")
print("=" * 60)

display(feature_metrics.head(10))


# -------------------------------------------------------------------------
# Individual Metric Views
# -------------------------------------------------------------------------
print("=" * 60)
print("Highest Monotonicity")
print("=" * 60)

display(
    feature_metrics[["Feature", "Monotonicity"]]
    .sort_values("Monotonicity", ascending=False).head(10)
)

print("=" * 60)
print("Highest Trendability")
print("=" * 60)

display(
    feature_metrics[["Feature", "Trendability"]]
    .sort_values("Trendability", ascending=False).head(10)
)

print("=" * 60)
print("Highest Prognosability")
print("=" * 60)

display(
    feature_metrics[["Feature", "Prognosability"]]
    .sort_values("Prognosability", ascending=False).head(10)
)

In [ ]:
# =============================================================================
# 13. Feature Redundancy Analysis
# =============================================================================
#
# Redundancy is Evidence.
#
# High correlation does NOT automatically remove either Feature.
#
print("=" * 60)
print("Feature Redundancy Analysis")
print("=" * 60)

feature_redundancy = find_redundant_features(
    df, feature_candidates,
    threshold=REDUNDANCY_THRESHOLD,
    method="spearman"
)

print(f"Threshold: |Spearman| >= {REDUNDANCY_THRESHOLD:.2f}")
print(f"Redundant pairs: {len(feature_redundancy):,}")

display(feature_redundancy)

In [ ]:
# =============================================================================
# 14. Global Feature ↔ Target Correlation
# =============================================================================
#
# 전체 Feature Candidate에 대해 Evidence를 계산한다.
#
# 단, Notebook 화면에서는 Cumulative Feature만 표시한다.
#
# 이유:
#   Raw / Rectified Feature의 열화 추세는 Cell 11에서 확인하고,
#   RDI Feature의 Target 관계를 이 단계에서 집중적으로 확인한다.
#
# 저장되는 CSV에는 전체 Feature Candidate의 결과가 보존된다.
#


print("=" * 60)
print("Global Feature ↔ Target Correlation")
print("=" * 60)


# -------------------------------------------------------------------------
# Full Evidence Calculation
# -------------------------------------------------------------------------

rul_correlation = calculate_rul_correlation(
    df,
    features=feature_candidates,
    targets=target_columns,
)


# -------------------------------------------------------------------------
# Display Subset
# -------------------------------------------------------------------------

available_cumulative = [
    feature for feature in CUMULATIVE_FEATURES
    if feature in feature_candidates
]


rul_correlation_display = (
    rul_correlation[rul_correlation["Feature"].isin(available_cumulative)]
    .assign(Abs_Spearman=lambda x: x["Spearman"].abs())
    .sort_values("Abs_Spearman", ascending=False)
)


print("Cumulative Feature ↔ Target Evidence:")
display(rul_correlation_display)

In [ ]:
# =============================================================================
# 15. Bearing-wise Feature ↔ Target Correlation
# =============================================================================
#
# 전체 Feature Candidate에 대해 Bearing-wise Evidence를 계산한다.
# 화면에는 Cumulative Feature만 표시한다.
#


print("=" * 60)
print("Bearing-wise Feature ↔ Target Correlation")
print("=" * 60)


# -------------------------------------------------------------------------
# Full Evidence Calculation
# -------------------------------------------------------------------------

rul_correlation_by_bearing = (
    calculate_rul_correlation_by_bearing(
        df,
        features=feature_candidates,
        targets=target_columns,
        bearing_col="bearing",
    )
)


# -------------------------------------------------------------------------
# Display Subset
# -------------------------------------------------------------------------

rul_correlation_by_bearing_display = (
    rul_correlation_by_bearing[
        rul_correlation_by_bearing["Feature"]
        .isin(available_cumulative)
    ]
)


display(rul_correlation_by_bearing_display)


# -------------------------------------------------------------------------
# Bearing-wise Spearman Summary
# -------------------------------------------------------------------------

bearing_spearman_summary = (
    rul_correlation_by_bearing
    .groupby(["Feature", "Target"], as_index=False)
    .agg(
        Mean_Spearman=("Spearman", "mean"),
        Std_Spearman=("Spearman", "std"),
        Min_Spearman=("Spearman", "min"),
        Max_Spearman=("Spearman", "max"),
    )
)


bearing_spearman_summary_display = (
    bearing_spearman_summary[
        bearing_spearman_summary["Feature"]
        .isin(available_cumulative)
    ]
)


print("=" * 60)
print("Bearing-wise Spearman Summary")
print("=" * 60)

display(bearing_spearman_summary_display)

In [ ]:
# =============================================================================
# 16. Detailed Feature ↔ Target Visualization
# =============================================================================
#
# 상세 Feature ↔ Target 산점도는 사용자가 명시한 Feature에 대해서만 생성한다.
#
# RUL_PLOT_FEATURES = []
#     → 그래프 생성 안 함
#
# RUL_PLOT_FEATURES = ["RDI_rms_vector"]
#     → 해당 Feature만 생성
#
# 이 셀은 Correlation Evidence를 대체하지 않는다.
# Correlation은 Cell 14/15에서 이미 계산된다.
#


print("=" * 60)
print("Feature ↔ Target Visualization")
print("=" * 60)


if not RUL_PLOT_FEATURES:
    print("No detailed Feature ↔ Target plots requested.")

else:
    if RUL_VISUALIZATION_TARGET not in target_columns:
        raise ValueError(
            "RUL_VISUALIZATION_TARGET is not "
            "a valid Target Candidate: "
            f"{RUL_VISUALIZATION_TARGET}"
        )


    invalid_plot_features = [
        feature for feature in RUL_PLOT_FEATURES
        if feature not in feature_candidates
    ]

    if invalid_plot_features:
        raise ValueError(
            "RUL_PLOT_FEATURES contains invalid "
            f"Feature Candidates: {invalid_plot_features}"
        )


    for feature in RUL_PLOT_FEATURES:
        print(f"\n[{feature}] vs [{RUL_VISUALIZATION_TARGET}]")

        fig = plot_feature_vs_target(
            df,
            feature=feature,
            target=RUL_VISUALIZATION_TARGET,
            bearing_col="bearing",
        )

        if SAVE_FIGURES:
            fig.savefig(
                os.path.join(
                    FIGURE_DIR,
                    f"feature_vs_"
                    f"{RUL_VISUALIZATION_TARGET}_"
                    f"{feature}.png",
                ),
                dpi=150, bbox_inches="tight",
            )

In [ ]:
# =============================================================================
# 17. Evidence Saving
# =============================================================================
#
# 화면에 표시하는 결과는 일부 Feature로 제한할 수 있지만,
# Evidence 파일에는 전체 Feature Candidate의 분석 결과를 저장한다.
#
# 따라서:
#
#   display(...)
#       → 목적별로 축소 가능
#
#   *.csv
#       → 전체 Evidence 보존
#
# Blueprint는 생성하지 않는다.
#


print("=" * 60)
print("Evidence Saving")
print("=" * 60)


if SAVE_EVIDENCE:
    os.makedirs(EVIDENCE_DIR, exist_ok=True)


    # ---------------------------------------------------------
    # Data Quality
    # ---------------------------------------------------------
    data_quality.to_csv(DATA_QUALITY_PATH, index=False)
    print(f"Saved: {DATA_QUALITY_PATH}")


    # ---------------------------------------------------------
    # Bearing Profile
    # ---------------------------------------------------------
    bearing_profile.to_csv(BEARING_PROFILE_PATH, index=False)
    print(f"Saved: {BEARING_PROFILE_PATH}")


    # ---------------------------------------------------------
    # Feature Metrics
    # ---------------------------------------------------------
    feature_metrics.to_csv(FEATURE_METRICS_PATH, index=False)
    print(f"Saved: {FEATURE_METRICS_PATH}")


    # ---------------------------------------------------------
    # Feature Redundancy
    # ---------------------------------------------------------
    feature_redundancy.to_csv(FEATURE_REDUNDANCY_PATH, index=False)
    print(f"Saved: {FEATURE_REDUNDANCY_PATH}")


    # ---------------------------------------------------------
    # Global RUL Correlation
    # ---------------------------------------------------------
    rul_correlation.to_csv(RUL_CORRELATION_PATH, index=False)
    print(f"Saved: {RUL_CORRELATION_PATH}")


    # ---------------------------------------------------------
    # Bearing-wise RUL Correlation
    # ---------------------------------------------------------
    rul_correlation_by_bearing.to_csv(RUL_CORRELATION_BY_BEARING_PATH, index=False)
    print(f"Saved: {RUL_CORRELATION_BY_BEARING_PATH}")

In [ ]:
# =============================================================================
# 18. EDA Evidence Summary
# =============================================================================

print("=" * 60)
print("EDA Evidence Summary")
print("=" * 60)


print(
    f"Feature Candidates          : "
    f"{len(feature_candidates):,}"
)

print(
    f"Target Candidates           : "
    f"{len(target_columns):,}"
)

print(
    f"Feature Metric rows         : "
    f"{len(feature_metrics):,}"
)

print(
    f"Redundant Feature pairs     : "
    f"{len(feature_redundancy):,}"
)

print(
    f"Global RUL correlation rows : "
    f"{len(rul_correlation):,}"
)

print(
    f"Bearing RUL rows            : "
    f"{len(rul_correlation_by_bearing):,}"
)


print("\n[Visualization Groups]")

print(
    f"Non-cumulative Features     : "
    f"{len(available_non_cumulative):,}"
)

for feature in available_non_cumulative:
    print(
        f"  - {feature}"
    )


print(
    f"Cumulative Features         : "
    f"{len(available_cumulative):,}"
)

for feature in available_cumulative:
    print(
        f"  - {feature}"
    )


print(
    f"RUL Plot Features            : "
    f"{len(RUL_PLOT_FEATURES):,}"
)

In [ ]:
# =============================================================================
# 19. Feature Selection — USER DECISION
# =============================================================================
#
# IMPORTANT
# ---------
# 여기서는 자동 선택을 수행하지 않는다.
# 위에서 생성한 Evidence를 검토한 후 사용자가 직접 입력한다.
#
# Feature Ranking Score는 추천 근거일 뿐이며,
# 높은 Score를 자동으로 Selected Feature로 변환하지 않는다.
#
# Whitelist 방식:
#
#     All Feature Candidates
#             ↓
#         EDA Evidence
#             ↓
#         User Decision
#             ↓
#     CANDIDATE_FEATURES
#
#
# Target도 동일하다.
#
#     Target Candidates
#             ↓
#         User Decision
#             ↓
#     SELECTED_TARGET
#
# -------------------------------------------------------------------------

CANDIDATE_FEATURES = [
    "RDI_rms", "Rectified_rms",
    "RDI_cf", "Rectified_cf",
    "RDI_p2p", "Rectified_p2p",
    "RDI_kurt", "Rectified_kurt",
    "temp_mean", "temp_max",
    "spectral_spread",
    "spectral_crest",
    "principal_energy_ratio",
]
SELECTED_TARGET = "RUL_norm"

SELECTION_CONFIG = {
    "candidate_features": CANDIDATE_FEATURES,
    "selected_target": SELECTED_TARGET
}

with open("outputs/03_eda_selection/selection_config.json", "w", encoding="utf-8") as f:
    json.dump(SELECTION_CONFIG, f, ensure_ascii=False, indent=4)

In [ ]:
# =============================================================================
# 20. Selection Validation
# =============================================================================
#
# Selection을 실제로 수행한 경우에만 검증한다.
#
# 빈 Selection은 이 Notebook의 현재 상태에서는 허용한다.
# 이는 EDA를 먼저 완료한 뒤 별도의 사용자 판단을 입력하기 위한 것이다.
#

print("=" * 60)
print("Selection Validation")
print("=" * 60)
# -------------------------------------------------------------------------
# Selected Feature Validation
# -------------------------------------------------------------------------
if CANDIDATE_FEATURES:
    invalid_features = [
        feature for feature in CANDIDATE_FEATURES
        if feature not in feature_candidates
    ]

    if invalid_features:
        raise ValueError(
            "Selected Feature(s) are not valid Feature Candidates: "
            f"{invalid_features}"
        )

    if len(CANDIDATE_FEATURES) != len(set(CANDIDATE_FEATURES)):
        raise ValueError("CANDIDATE_FEATURES contains duplicates.")

    print("Selected Features:")
    for feature in CANDIDATE_FEATURES:
        print(f"  - {feature}")

else:
    print("Selected Features: [not specified]")

# -------------------------------------------------------------------------
# Selected Target Validation
# -------------------------------------------------------------------------
if SELECTED_TARGET is not None:
    if SELECTED_TARGET not in target_columns:
        raise ValueError(
            "SELECTED_TARGET is not a valid Target Candidate: "
            f"{SELECTED_TARGET}"
        )
    print(f"Selected Target: {SELECTED_TARGET}")

else:
    print("Selected Target: [not specified]")

In [ ]:
# =============================================================================
# 21. Final Selection Summary
# =============================================================================
print("=" * 60)
print("03 EDA & Feature Selection Summary")
print("=" * 60)


print("\n[Input]")
print(f"Engineered Learning Dataset: {LEARNING_INPUT_PATH}")

print(f"Rows                       : {len(df):,}")

print(f"Feature Candidates         : {len(feature_candidates):,}")
print(f"Target Candidates          : {len(target_columns):,}")


print("\n[Evidence]")
print(f"Feature Metrics            : {FEATURE_METRICS_PATH}")
print(f"Feature Redundancy         : {FEATURE_REDUNDANCY_PATH}")

print(f"Global RUL Correlation     : {RUL_CORRELATION_PATH}")
print(
    "Bearing RUL Correlation    : "
    f"{RUL_CORRELATION_BY_BEARING_PATH}"
)

print("\n[Selection]")
if CANDIDATE_FEATURES:
    print("Selected Features:")
    for feature in CANDIDATE_FEATURES:
        print(f"  - {feature}")
else:
    print("Selected Features: [not specified]")


print(
    "Selected Target: "
    f"{SELECTED_TARGET if SELECTED_TARGET is not None else '[not specified]'}"
)


print("\n03 EDA & Feature Selection complete.")